# What Drives National Happiness?
### A World Happiness Report 2021 Analysis


# Introduction

This project analyzes the **World Happiness Report** using two complementary datasets:

- a country-level dataset for **2021**;
- a panel dataset covering multiple years from **2005 to 2020**.

The goal is to understand which economic, social, health-related, and institutional factors are most strongly associated with national happiness. The analysis also compares countries across regions and broader continent groups, then uses the panel dataset to observe time trends and emotional indicators.


# Research Questions

1. Which factors are most strongly associated with happiness in 2021?
2. How does happiness vary across countries, regions, and broader continent groups?
3. How has happiness changed over time for selected countries?
4. How are positive and negative emotional indicators related to the happiness score?


# Project Structure

The notebook is organized into four main parts:

- data inspection and cleaning;
- descriptive analysis of happiness in 2021;
- regional and continent-based comparison;
- relationship analysis between happiness and key explanatory variables, including time-based and emotional indicators from the panel dataset.


# Analysis Part 1: Structured Data Analysis


## Data Inspection and Cleaning

The two datasets are loaded into consistently named DataFrames:

- `df_2021` contains the country-level data for 2021;
- `df_panel` contains the multi-year panel data.

Column names are harmonized so that both datasets can be used with the same variable names wherever possible.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

candidate_dirs = [
    Path("/kaggle/input/datasets/ajaypalsinghlo/world-happiness-report-2021"),
    Path("/kaggle/input/world-happiness-report-2021"),
    Path.cwd(),
    Path.cwd() / "data",
    Path.home() / "Downloads",
]


def find_data_file(filename):
    for data_dir in candidate_dirs:
        path = data_dir / filename
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Put the file in the notebook folder, "
        "in a data folder, in Downloads, or update candidate_dirs."
    )

path_2021 = find_data_file("world-happiness-report-2021.csv")
path_panel = find_data_file("world-happiness-report.csv")

df_2021 = pd.read_csv(path_2021)
df_panel = pd.read_csv(path_panel)

display(df_2021.head())
display(df_panel.head())


In [ ]:
columns_to_drop = [
    "Standard error of ladder score",
    "upperwhisker",
    "lowerwhisker",
    "Ladder score in Dystopia",
    "Explained by: Log GDP per capita",
    "Explained by: Generosity",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Perceptions of corruption",
    "Dystopia + residual",
]

df_2021 = df_2021.drop(columns=columns_to_drop, errors="ignore")

df_2021 = df_2021.rename(columns={
    "Logged GDP per capita": "GDP per capita",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.rename(columns={
    "Life Ladder": "Ladder score",
    "Log GDP per capita": "GDP per capita",
    "Healthy life expectancy at birth": "Healthy life expectancy",
    "Freedom to make life choices": "Life choices freedom",
})

df_panel = df_panel.dropna(subset=["Ladder score", "year"])

display(df_2021.info())
display(df_panel.info())


### Observation

For the purposes of this analysis, the 2021 dataset keeps only the main explanatory variables: GDP per capita, social support, healthy life expectancy, freedom to make life choices, generosity, and perceptions of corruption. Variables directly derived from the happiness score are removed to avoid circular interpretation.


In [ ]:
quality_summary = pd.DataFrame({
    "duplicates": [df_2021.duplicated().sum(), df_panel.duplicated().sum()],
    "missing_values_total": [df_2021.isnull().sum().sum(), df_panel.isnull().sum().sum()],
    "rows": [len(df_2021), len(df_panel)],
    "columns": [df_2021.shape[1], df_panel.shape[1]],
}, index=["df_2021", "df_panel"])

quality_summary


### Insight

The 2021 dataset is clean after removing derived variables. The panel dataset still contains some missing values in explanatory variables, which is common in multi-year international datasets. Rows with missing `Ladder score` or `year` are removed because those two variables are essential for the time-based analysis.


## Data Visualization


### 1. Global Distribution of Happiness Scores

This chart shows how happiness scores are distributed across the countries included in the 2021 dataset.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df_2021, x="Ladder score", bins=15, kde=True)
plt.title("Global Distribution of Happiness Scores in 2021")
plt.xlabel("Ladder score")
plt.ylabel("Number of countries")
plt.show()


#### Insight

The distribution is concentrated around the middle range, with most countries scoring between roughly 4.5 and 6.5. Very high and very low scores are less frequent, suggesting that the global pattern is mostly continuous rather than sharply divided into separate groups.


### 2. Top 10 and Bottom 10 Countries

This section identifies the countries with the highest and lowest happiness scores in 2021.


In [ ]:
top_10 = df_2021.sort_values("Ladder score", ascending=False).head(10)
bottom_10 = df_2021.sort_values("Ladder score", ascending=True).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_10, x="Ladder score", y="Country name", hue="Country name", palette="Greens_r", legend=False)
plt.title("10 Happiest Countries in 2021")
plt.xlabel("Ladder score")
plt.ylabel("Country")
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=bottom_10, x="Ladder score", y="Country name", hue="Country name", palette="Reds_r", legend=False)
plt.title("10 Least Happy Countries in 2021")
plt.xlabel("Ladder score")
plt.ylabel("Country")
plt.show()


#### Insight

The highest-ranking countries are mostly European, with small score differences within the top group. The lowest-ranking countries are more concentrated in regions facing stronger economic, health, and institutional challenges. This contrast points to large global inequalities in well-being.


### 3. Regional Distribution of Top and Bottom Countries

This chart compares the regional composition of the top 10 and bottom 10 countries.


In [ ]:
top_10 = top_10.copy()
bottom_10 = bottom_10.copy()
top_10["Group"] = "Top 10"
bottom_10["Group"] = "Bottom 10"
combined = pd.concat([top_10, bottom_10], ignore_index=True)

plt.figure(figsize=(12, 6))
palette = {"Top 10": "seagreen", "Bottom 10": "firebrick"}
sns.countplot(data=combined, x="Regional indicator", hue="Group", palette=palette)
plt.title("Regional Distribution of Top and Bottom 10 Countries")
plt.xlabel("Region")
plt.ylabel("Number of countries")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


#### Insight

The top group is heavily concentrated in Western Europe and North America/ANZ. The bottom group is dominated by Sub-Saharan Africa, with some representation from South Asia and the Middle East and North Africa. Happiness therefore appears strongly connected to regional context.


### 4. Regional Distribution of Happiness Scores

The regional boxplot adds more information than the top/bottom comparison because it shows central values, dispersion, and outliers for every region.


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_2021, x="Regional indicator", y="Ladder score")
plt.title("Distribution of Happiness Scores by Region")
plt.xlabel("Region")
plt.ylabel("Ladder score")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


#### Insight

Western Europe has high and relatively stable scores, while Sub-Saharan Africa has lower scores and greater vulnerability. Other regions, such as Asia and Latin America, show more mixed distributions. This makes the boxplot valuable because it goes beyond ranking and reveals within-region variation.


### 5. Continent-Level Comparison

Regions are grouped into broader continent categories to simplify the geographic comparison. This regrouping is analytical rather than strictly geographic.


In [ ]:
def continent_from_region(region):
    if region in ["Western Europe", "Eastern Europe"]:
        return "Europe"
    if region in ["Commonwealth of Independent States"]:
        return "Eurasia"
    if region in ["South Asia", "East Asia", "Southeast Asia"]:
        return "Asia"
    if region in ["North America and ANZ"]:
        return "North America & Oceania"
    if region in ["Latin America and Caribbean"]:
        return "South America"
    if region in ["Middle East and North Africa", "Sub-Saharan Africa"]:
        return "Africa"
    return "Other"

df_2021["Continent"] = df_2021["Regional indicator"].apply(continent_from_region)

df_2021[["Country name", "Regional indicator", "Continent"]].head()


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_2021, x="Continent", y="Ladder score")
plt.title("Distribution of Happiness Scores by Continent Group")
plt.xlabel("Continent group")
plt.ylabel("Ladder score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


#### Insight

The continent grouping makes the global pattern easier to read. Europe and North America/Oceania tend to have higher happiness scores, while Africa is generally lower. Asia and South America occupy more middle-range positions, with greater internal diversity.


### 6. GDP per Capita and Happiness by Continent

This scatterplot is kept from the second notebook because it adds analytical value: it shows the relationship between income and happiness while also revealing geographic clusters.


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_2021, x="GDP per capita", y="Ladder score", hue="Continent")
plt.title("GDP per Capita and Happiness by Continent")
plt.xlabel("GDP per capita")
plt.ylabel("Ladder score")
plt.legend(title="Continent", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


#### Insight

There is a clear positive association between GDP per capita and happiness. European countries are generally located in the higher-GDP and higher-happiness area, while many African countries appear in the lower part of both dimensions. However, the scatterplot also shows that income alone does not fully explain happiness.


### 7. Correlation Between Happiness and Key Variables

This analysis identifies which explanatory variables are most strongly associated with the 2021 happiness score.


In [ ]:
corr = df_2021.corr(numeric_only=True)
ladder_corr = corr["Ladder score"].drop("Ladder score").sort_values(ascending=False)

display(ladder_corr)

plt.figure(figsize=(12, 6))
colors = ["steelblue" if value > 0 else "firebrick" for value in ladder_corr]
ax = sns.barplot(x=ladder_corr.index, y=ladder_corr.values, hue=ladder_corr.index, palette=colors, legend=False)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")
plt.title("Correlation Between Happiness and Key Variables")
plt.xlabel("Variables")
plt.ylabel("Correlation coefficient")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
variables = ["GDP per capita", "Healthy life expectancy", "Social support"]

plt.figure(figsize=(18, 6))
for i, var in enumerate(variables, 1):
    plt.subplot(1, 3, i)
    sns.regplot(
        data=df_2021,
        x=var,
        y="Ladder score",
        scatter_kws={"alpha": 0.6},
        line_kws={"color": "red"},
    )
    plt.title(f"{var} vs Happiness")
    plt.xlabel(var)
    plt.ylabel("Ladder score")

plt.tight_layout()
plt.show()


#### Insight

GDP per capita, healthy life expectancy, and social support are the strongest positive correlates of happiness. Freedom to make life choices is also positively associated with happiness, while perceptions of corruption show a negative relationship. These results suggest that well-being is shaped by a combination of economic security, health, social connection, and institutional trust.


### 8. Standardized Comparison of Happiness Factors by Continent

This chart is included because standardization allows variables measured on different scales to be compared more fairly across continent groups.


In [ ]:
factors = [
    "GDP per capita",
    "Social support",
    "Healthy life expectancy",
    "Life choices freedom",
    "Generosity",
    "Perceptions of corruption",
]

df_std = (df_2021[factors] - df_2021[factors].mean()) / df_2021[factors].std()
df_std["Continent"] = df_2021["Continent"]
df_continent_std = df_std.groupby("Continent").mean()

df_continent_std.plot(kind="bar", figsize=(12, 6))
plt.title("Standardized Happiness Factors by Continent")
plt.xlabel("Continent group")
plt.ylabel("Average standardized value")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Factors", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


#### Insight

The standardized comparison highlights structural differences between continent groups. Higher-scoring groups tend to perform better on GDP per capita, healthy life expectancy, and social support. Lower-scoring groups often fall below the global average on several of these factors at the same time, which suggests that happiness inequalities are multidimensional rather than driven by a single variable.


## Time-Based Analysis Using the Panel Dataset


### 9. Correlation Matrix for the Panel Dataset

The panel dataset includes additional time-based and emotional variables. The heatmap is useful because it summarizes how these variables relate to each other across years.


In [ ]:
panel_corr = df_panel.select_dtypes(include="number").corr()

plt.figure(figsize=(10, 8))
sns.heatmap(panel_corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix for the Panel Dataset")
plt.tight_layout()
plt.show()


#### Insight

The panel data confirms the strong positive relationship between happiness, GDP per capita, social support, and healthy life expectancy. Negative affect and perceptions of corruption are negatively associated with happiness, which is consistent with the 2021 analysis.


### 10. Happiness Trends for Selected Countries

France, the United States, and China are selected to compare happiness trends across countries with different social, economic, and political contexts.


In [ ]:
selected_countries = ["France", "United States", "China"]
df_selected = df_panel[df_panel["Country name"].isin(selected_countries)]

plt.figure(figsize=(10, 5))
sns.lineplot(data=df_selected, x="year", y="Ladder score", hue="Country name", marker="o")
plt.title("Happiness Trends Over Time")
plt.xlabel("Year")
plt.ylabel("Ladder score")
plt.tight_layout()
plt.show()


#### Insight

The trend comparison shows that happiness does not evolve in the same way across countries. Longitudinal data is therefore important because it captures changes that a single-year ranking cannot show.


### 11. Positive Affect, Negative Affect, and Happiness

These scatterplots add value because they connect the happiness score to emotional indicators, not only to economic or institutional variables.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_panel, x="Positive affect", y="Ladder score", alpha=0.5, ax=axes[0])
axes[0].set_title("Positive Affect and Happiness")
axes[0].set_xlabel("Positive affect")
axes[0].set_ylabel("Ladder score")

sns.scatterplot(data=df_panel, x="Negative affect", y="Ladder score", alpha=0.5, ax=axes[1])
axes[1].set_title("Negative Affect and Happiness")
axes[1].set_xlabel("Negative affect")
axes[1].set_ylabel("Ladder score")

plt.tight_layout()
plt.show()


#### Insight

Positive affect is generally associated with higher happiness scores, while negative affect is associated with lower scores. This supports the idea that national happiness is not only linked to material conditions, but also to subjective emotional experience.


# Summary

This project shows that national happiness is strongly associated with a combination of economic, health, social, and institutional factors. In 2021, GDP per capita, healthy life expectancy, and social support have the strongest positive relationships with happiness. Regional and continent-level visualizations reveal substantial geographic inequalities, with Western Europe and North America/Oceania generally scoring higher and Sub-Saharan Africa scoring lower.

The panel dataset adds a time-based perspective. It confirms that happiness is connected to structural indicators such as GDP, health, and social support, while also showing the importance of emotional variables. Overall, the analysis suggests that happiness is multidimensional: no single factor fully explains it, but countries with stronger economic security, healthier populations, stronger social support, and greater institutional trust tend to report higher well-being.
